# Plotting with PyQTgraph

Before starting, please install *PyQT Graph* tools
```bash
    > pip install pyqt6 pyqtgraph
``` 

## Plotting signals from remote antenna

In [7]:
# Load packages

import numpy as np
import time
import matplotlib.pyplot as plt
from megamicros.log import log
from megamicros.core.ws import MemsArrayWS

log.setLevel( "INFO" )

# Set server access credentials
#HOST = 'buzenval20.fr'
#HOST = 'parisparc.biimea.tech'
HOST = 'localhost'
PORT = 9002

import pyqtgraph as pg
from pyqtgraph.Qt import QtCore


## Connecting to the remote server

Providing a *MBS* server is running at ``HOST:PORT``, one can try to connect by creating a ``MemsArrayWS`` object.

In [8]:
# Define the antenna
try:
    antenna = MemsArrayWS( HOST, port=PORT )
except Exception as e:
    print( f"Failed: {e}" )

2023-11-22 22:39:15,925 [INFO]:  .Install MemsArrayWS settings
2023-11-22 22:39:15,928 [INFO]:  .Created a new antenna
2023-11-22 22:39:15,929 [INFO]:  .Async event loop already running. Adding coroutine to the event loop...


2023-11-22 22:39:15,934 [INFO]:  .Try connecting to ws://localhost:9002...
2023-11-22 22:39:15,942 [INFO]:  .Received positive answer from server
2023-11-22 22:39:15,943 [INFO]:  .Getting settings values from remote receiver...
2023-11-22 22:39:15,947 [ERROR]: in megamicros.log (ws.py:238): Server connection failed: Unable to get settings from server: No antenna connected
2023-11-22 22:39:15,948 [ERROR]: in megamicros.log (ws.py:200): Unable to connect to remote server localhost:9002


## Plotting DB signals

One has only to specify the file identifier.

The ``MemsArrayDB`` constructor connects to the database and populates the antenna parameters with metadata received from the database. An exception is raised if connection failed.

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import queue
from megamicros.log import log
from megamicros.core.db import MemsArrayDB

log.setLevel( "INFO" )

# Set database access credentials
#DBHOST = 'http://dbwelfare.biimea.io/'
DBHOST = 'http://dbchantier.biimea.io/'
LOGIN = 'ailab'
EMAIL = 'bruno.gas@biimea.com'
PASSWORD = '#T;uZnQ5UJ_JC~&'

In [4]:
# Choose the a signal file of the database and the MEMs to plot
FILE_ID = 1239
FILE_ID = 1231
MEMS = [0, 1, 2, 3, 4, 5, 6],

# Define the antenna
antenna = MemsArrayDB( 
    dbhost=DBHOST, login=LOGIN, email=EMAIL, password=PASSWORD, 
    file_id=FILE_ID
)

2023-11-22 21:30:28,839 [INFO]:  .Install MemsArrayDB settings
2023-11-22 21:30:28,848 [INFO]:  .Created a new antenna
2023-11-22 21:30:28,849 [INFO]:  .Install MemsArrayDB settings
2023-11-22 21:30:28,850 [INFO]:  .Try connecting on endpoint database http://dbchantier.biimea.io//dj-rest-auth/login/...
2023-11-22 21:30:29,342 [INFO]:  .Got HTTP 200 status code from server
2023-11-22 21:30:29,343 [INFO]:  .Received CSRF token: tYiPa8VGBVS2FYlNSZg41ohqQ4teTEmh. Update session with
2023-11-22 21:30:29,344 [INFO]:  .Received session id: 9karvb028lpbhrlnb4wucnm6umgfmw0d
2023-11-22 21:30:29,345 [INFO]:  .Successfully connected on http://dbchantier.biimea.io/
2023-11-22 21:30:29,349 [INFO]:  .Downloading metadata for object 'sourcefile' [1231]...
2023-11-22 21:30:29,350 [INFO]:  .Send a database request on endpoint: http://dbchantier.biimea.io//sourcefile/1231
2023-11-22 21:30:29,382 [INFO]:  .Object sourcefile found with identifier [1231] 
2023-11-22 21:30:29,383 [INFO]:  .Set 24 available M

## Init graph

Get data as `float32` type. 

In [ ]:
# init PyQtgraph
win = pg.GraphicsLayoutWidget(show=True, title="Plotting database signals")
win.resize(1000,600)
win.setWindowTitle('Plotting database signals')
pg.setConfigOptions(antialias=True)
graph = win.addPlot(title="Microphones")
graph.setYRange(-5,5, padding=0, update = False)
curves = []
for s in range( len( MEMS ) ):
    curves.append( graph.plot(pen='y' ) )

# Define the plot function
# Get last queued signal and plot it
def plot_on_the_fly( antenna, curves ):

	try:
		data = antenna.signal_q.get( timeout=1 )
	except queue.Empty:
		return

	t = np.arange( np.size( data, 1 ) )/antenna.sampling_frequency
	for s in range( antenna.mems_number ):
		curves[s].setData( t, ( data[s,:] * antenna.sensibility ) + s - antenna.mems_number/2 )


timer = QtCore.QTimer()
timer.timeout.connect( lambda: plot_on_the_fly( antenna, curves ) )

In [ ]:
# Run antenna
antenna.run(
    mems = MEMS,
    start = 120,
    duration=10,
    counter_skip = True,
    datatype='float32',
)

# Init a np.ndarray
signals = np.ndarray( (0, antenna.channels_number ) )

# Get signals
for data in antenna:
    signals = np.concatenate( ( signals, data ), axis=0 )

# waiting for the end of the running thread is mandatory
antenna.wait()
print( f"exit from loop. Signal shape is: {np.shape( signals )}" )

In [ ]:
# halt server
antenna.shutdown()

2023-11-22 22:39:36,620 [INFO]:  .Connecting to remote host localhost:9002...
2023-11-22 22:39:36,623 [INFO]:  .Connected
2023-11-22 22:39:36,625 [INFO]:  .Send shutdown command to server
2023-11-22 22:39:36,628 [INFO]:  .Remote server shutdown success
